#Part 1 — Python & Data Foundations: ingest, validate, clean, enrich

##Task 1.1 — Read the file the foundations way

Using only the csv module (not pandas): open appointments.csv with with open(..., newline=""), read it with csv.DictReader, and print the header and row count. Write a function parse_row(row) that converts age, fee_inr and prior_no_shows to int and distance_km to float, returning the converted dictionary. Loop over every row; use try/except ValueError to put rows that fail into a rejected list and the rest into an accepted list. Save rejected with json.dump(..., indent=2) as rejected_rows.json.

In [ ]:
import csv
import json


'''
  The below function converts the columns 'age', 'fee_inr' and 'prior_no_shows'
  to int data type and 'distance_km' into float data type. If the value being
  tried to converted is not a matching data type, then the TypeError will be
  caught and a flag 'isRejected' will be set to True to indicate that the value
  for the column is not of int or float.

  This function returns the isRejected flag and the row back to the caller.
'''
def parse_row(row):

  isRejected = False
  parsed_row = row.copy()

  try:
    parsed_row['age'] = int(row['age'])
  except (ValueError, TypeError):
    isRejected = True

  try:
    parsed_row['fee_inr'] = int(row['fee_inr'])
  except (ValueError, TypeError):
    isRejected = True

  try:
    parsed_row['prior_no_shows'] = int(row['prior_no_shows'])
  except (ValueError, TypeError):
    isRejected = True

  try:
    parsed_row['distance_km'] = float(row['distance_km'])
  except (ValueError, TypeError):
    isRejected = True

  return isRejected, parsed_row


'''
  This function creates a file with the given 'filename' and writes the data
  provided in the 'data' variable, which should be a list of dictionary objects
  into the file.
'''
def write_to_jsonfile(filename, data):
  with open(filename, mode='w', encoding='UTF-8') as file:
    json.dump(data, file, indent=2, ensure_ascii=False)

  print('Data has been written successfully into the file.')


# ******
# Step 1: Read the content from the csv file using csv.DictReader function
# ******
with open('/content/appointments.csv',mode='r',newline="",encoding='UTF-8') as file:
  appointments_csv= csv.DictReader(file)

  print('Headers from the \'appointments.csv\': ')
  headers = appointments_csv.fieldnames

  # ******
  # Step 2: Print column headers
  # ******
  for index, header_col in enumerate(headers, start=1):
    print(f'{index:>2}.  {header_col}')

  #Convering the appointments_csv to a list
  appointments_list = list(appointments_csv)

print('\n')
# ******
# Step 3: Print the row count
# ******
print(f'Row Count: {len(appointments_list)}')


print('\n')
rejected_list = []
accepted_list = []

# ******
# Step 4: Convert the datatypes of select columns into int and float accordingly
# ******
# Call the parse_row for each row in the dataset
for row in appointments_list:

  isRejected, parsed_row = parse_row(row)

  if isRejected:
    rejected_list.append(parsed_row)
  else:
    accepted_list.append(parsed_row)

write_to_jsonfile('rejected_rows.json', rejected_list)


Headers from the 'appointments.csv': 
 1.  appointment_id
 2.  patient_id
 3.  clinic_id
 4.  booked_on
 5.  appt_date
 6.  age
 7.  gender
 8.  distance_km
 9.  fee_inr
10.  booking_channel
11.  visit_type
12.  reminder_sent
13.  prior_no_shows
14.  status


Row Count: 1020


Data has been written successfully into the file.


##Task 1.2 — Inspect and clean, pass 1

Load the file with pd.read_csv. Print shape, info(), isnull().sum() and duplicated().sum(). Then, in this order: (a) drop exact duplicate rows; (b) clean booking_channel with .str.strip().str.title(); (c) convert appt_date with pd.to_datetime(..., errors="coerce") and booked_on with pd.to_datetime; (d) drop rows where appt_date is missing after conversion or status is missing. Print the row count after each step. This task starts again from the original appointments.csv: Task 1.1 is a separate exercise in reading with the csv module, so do not use its accepted/rejected lists here — the pandas cleaning in 1.2–1.3 is the one that feeds the rest of the project.

In [ ]:
import pandas as pd
import warnings


warnings.filterwarnings('ignore')

# ******
# Step 1: Read the content from the csv file using pd.read_csv
# ******
appointments_df = pd.read_csv('/content/appointments.csv')

# ******
# Step 2: Print shape, info(), isnull().sum() and duplicated().sum()
# ******
print(f'Shape: {appointments_df.shape}')

print('\n')
print(f'Info: \n{appointments_df.info}')

print('\n')
print(f'Total Number of rows will NULL values: \n{appointments_df.isnull().sum()}')

print('\n')
print(f'Total Number of rows will Duplicated rows: {appointments_df.duplicated().sum()}')

Shape: (1020, 14)


Info: 
<bound method DataFrame.info of      appointment_id patient_id clinic_id   booked_on   appt_date   age gender  \
0             A0001       P509       C05  2025-01-24  2025-01-25  60.0      F   
1             A0002       P046       C07  2025-05-27  2025-06-06  39.0      M   
2             A0003       P009       C02  2025-03-24  2025-03-30  63.0      F   
3             A0003       P009       C02  2025-03-24  2025-03-30  63.0      F   
4             A0004       P482       C01  2025-04-18  2025-04-22  52.0      F   
...             ...        ...       ...         ...         ...   ...    ...   
1015          A0996       P415       C05  2025-01-09  2025-01-31  20.0      M   
1016          A0997       P247       C05  2025-02-23  2025-02-27  77.0      M   
1017          A0998       P422       C06  2025-05-05  2025-05-16  58.0      M   
1018          A0999       P509       C05  2025-03-30  2025-04-12  59.0      F   
1019          A1000       P085       C08  2024-12-

In [ ]:
# ******
# Step 3a: drop exact duplicate rows
# ******
df_clean = appointments_df.drop_duplicates()

print(f'Row count after removing Duplicates: {len(df_clean)}')

# ******
# Step 3b: clean booking_channel with .str.strip().str.title()
# ******

df_clean['booking_channel'] = df_clean.loc[:, 'booking_channel'].str.strip().str.title()

print(f'Row count after cleaning the column \'booking_channel\': {len(df_clean)}')

# ******
# Step 3c: convert appt_date with pd.to_datetime(..., errors="coerce") and booked_on with pd.to_datetime
# ******
df_clean['appt_date'] = pd.to_datetime(df_clean['appt_date'],errors='coerce')
df_clean['booked_on'] = pd.to_datetime(df_clean['booked_on']) #,errors='coerce')

print(f'Row count after converting date columns: {len(df_clean)}')

# ******
# Step 3d: drop rows where appt_date is missing after conversion or status is missing
# ******
df_clean = df_clean.dropna(subset=['appt_date','status'])

print(f'Row count after dropping null value rows: {len(df_clean)}')


Row count after removing Duplicates: 1000
Row count after cleaning the column 'booking_channel': 1000
Row count after converting date columns: 1000
Row count after dropping null value rows: 990


## Task 1.3 — Clean, pass 2: validity, gaps and outliers

Work in this order, so that the quartiles and fences in (c) are computed after the gaps are filled. (a) Treat any age below 1 or above 100 as invalid and set it to missing. (b) Fill missing age with the median (then .round().astype(int)) and missing distance_km with its median. (c) For distance_km and fee_inr, compute Q1, Q3 and IQR with .quantile, and cap values at Q3 + 1.5×IQR and Q1 − 1.5×IQR with np.clip. (d) Round distance_km to 1 decimal and fee_inr to whole rupees (.round().astype(int)). Print how many values you capped in each column.

In [ ]:
import numpy as np

# ******
# Step 1a: drop rows where appt_date is missing after conversion or status is
#          missing
# ******
df_clean.loc[(df_clean['age'] < 1) | (df_clean['age'] > 100),'age'] = np.nan



# ******
# Step 1b: Fill missing age with the median (then .round().astype(int)) and
#          missing distance_km with its median.
# ******
print(f"Total number of age values being replaced: {df_clean['age'].isna().sum()}")

median_age = df_clean['age'].median()

print(f"Median Age: {median_age}")

df_clean.loc[(df_clean['age'].isna()), 'age'] = median_age

df_clean['age'] = (df_clean['age']).round().astype(int)



dist_km_median = df_clean['distance_km'].median()

print(f"Median Distance Kms: {dist_km_median}")

df_clean.loc[(df_clean['distance_km'].isna()), 'distance_km'] = dist_km_median



# ******
# Step 1c: For distance_km and fee_inr, compute Q1, Q3 and IQR with .quantile,
#          and cap values at the Upper and Lower limit
# ******
def calculate_limits(data, col_name):

  # print(df_clean[col_name])

  Q1 = data[col_name].quantile(0.25)
  Q3 = data[col_name].quantile(0.75)

  IQR = Q3 - Q1

  upper_limit = Q3 + 1.5*IQR
  lower_limit = Q1 - 1.5*IQR

  return upper_limit, lower_limit

# Cap the values for distance_km
dist_km_upper_lmt, dist_km_lower_lmt = calculate_limits(df_clean,'distance_km')


dist_km_capped_count = ((df_clean['distance_km'] > dist_km_upper_lmt).sum()) + ((df_clean['distance_km'] < dist_km_lower_lmt).sum())
print(f'Total Number of capped values of \'distance_km\': {dist_km_capped_count}')


df_clean.loc[(df_clean['distance_km'] > dist_km_upper_lmt), 'distance_km'] = dist_km_upper_lmt
df_clean.loc[(df_clean['distance_km'] < dist_km_lower_lmt), 'distance_km'] = dist_km_lower_lmt

# Cap the values for fee_inr
fee_inr_upper_lmt, fee_inr_lower_lmt = calculate_limits(df_clean,'fee_inr')

fee_inr_capped_count = ((df_clean['fee_inr'] > fee_inr_upper_lmt).sum()) + ((df_clean['fee_inr'] < fee_inr_lower_lmt).sum())
print(f'Total Number of capped values of \'fee_inr\': {fee_inr_capped_count}')

df_clean.loc[(df_clean['fee_inr'] > fee_inr_upper_lmt), 'fee_inr'] = fee_inr_upper_lmt
df_clean.loc[(df_clean['fee_inr'] < fee_inr_lower_lmt), 'fee_inr'] = fee_inr_lower_lmt

# ******
# Step 1d: Round distance_km to 1 decimal and fee_inr to whole rupees (.round().astype(int))
# ******
df_clean['distance_km'] = df_clean['distance_km'].round(1)
df_clean['fee_inr'] = df_clean['fee_inr'].round().astype(int)

Total number of age values being replaced: 17
Median Age: 48.0
Median Distance Kms: 8.25
Total Number of capped values of 'distance_km': 26
Total Number of capped values of 'fee_inr': 3


## Task 1.4 — Features and a NumPy summary

Create lead_days = days between booked_on and appt_date (use .dt.days) and no_show = 1 if status is "No-Show" else 0. Then take X = df[["age","distance_km","fee_inr","lead_days"]].to_numpy() and, without loops, compute: (a) the mean of each column (axis=0); (b) the number and share of rows with lead_days > 14 using a boolean mask (mask.sum() counts the True values and mask.mean() gives the share); (c) z-scores of the distance_km column using (x - np.mean(x)) / np.std(x) and the count of rows with |z| > 2.

In [ ]:
# Step 1: Calculate the lead days
df_clean['lead_days'] = (df_clean['appt_date'] - df_clean['booked_on']).dt.days

# Step 2: Evaluate and store the no_show values
df_clean['no_show'] = (df_clean['status'] == 'No-Show').astype(int)

print(f"Total number of No-Shows: {(df_clean['status'] == 'No-Show').sum()}")

# Step 3: Create a numpy array of selected columns
df_clean_subst = df_clean[['age','distance_km','fee_inr','lead_days']].to_numpy()

# Step 4a: The mean of each column
print(f'Mean of the selected columns: {df_clean_subst.mean(axis=0)}')

# Step 4b: the number and share of rows with lead_days > 14
mask = (df_clean_subst[:,3] > 14)

print('\n')
print(f'Total number of rows filtered with lead_days gt 14: {mask.sum()}')
print(f'Mean of rows filtered with lead_days gt 14: {mask.mean():.3f}')

# Step 4c: Z-score for the column 'distance_km'
#          and count of rows with more than |z| > 2
dist_km_values = df_clean_subst[:,1]

mean = dist_km_values.mean()
std = dist_km_values.std()

z_scores = (dist_km_values - mean) / std

mask_z_score = dist_km_values > np.abs(z_scores)

print('\n')
print(f'Total rows with Z-Score more than 2: {mask_z_score.sum()}')

Total number of No-Shows: 218
Mean of the selected columns: [ 48.17373737   9.50090909 635.71414141  11.81919192]


Total number of rows filtered with lead_days gt 14: 296
Mean of rows filtered with lead_days gt 14: 0.299


Total rows with Z-Score more than 2: 957


## Task 1.5 — Enrich the clinics from a live API

For each of the 8 clinics call the free Open-Meteo API with requests.get("https://api.open-meteo.com/v1/forecast", params={"latitude": ..., "longitude": ..., "hourly": "temperature_2m", "forecast_days": 1}). Check status_code == 200 before using the reply, read the JSON, take the first value of ["hourly"]["temperature_2m"] (the midnight-UTC reading, a stand-in rather than the current weather) as temp_c, and save clinics_enriched.csv (the original columns plus temp_c; df.to_csv("file.csv", index=False) writes a DataFrame to CSV). No API key is needed. If the API fails: pass timeout=10 to requests.get, wrap the call in try/except requests.exceptions.RequestException, retry that clinic once, and if it still fails (or the status is not 200, or hourly is missing) store None as temp_c, print the clinic's name and the reason, and carry on. Never type in a made-up temperature; in load_data.sql a missing value is written as NULL (unquoted).

In [ ]:
import requests
from requests.exceptions import RequestException
import pandas as pd


def get_forecast(row):

  url = 'https://api.open-meteo.com/v1/forecast'
  params = {
      "latitude": row['latitude'],
      "longitude": row['longitude'],
      "hourly": "temperature_2m",
      "forecast_days": 1
  }

  temp_c = None

  try:

    # Call the API
    response = requests.get(url,params=params)

    # Check if response status is Success
    if (response.status_code == 200):

      data = response.json()

    #  If response status is not Success, retry with a timeout=10
    else:

      retry_response = requests.get(url,params=params,timeout=10)

      if(retry_response.status_code == 200):

        data = retry_response.json()

      else:

        return None

    temp_c = data['hourly']['temperature_2m'][0]

  except RequestException as re:
      print(f'Network or API request failed for the clinic {row['clinic_name']}: {re}')
      temp_c = None

  except KeyError as ke:
      print(f'The API response for {row['clinic_name']} does not have the \'hourly\' element: {ke}')
      temp_c = None

  return temp_c


# Read the clinic.csv file
clinic_df = pd.read_csv('/content/clinics.csv')

# Get the midnight temperature for the area of the clinic and set into the dataframe
clinic_df['temp_c'] = clinic_df.apply(get_forecast,axis=1)

# Write this dataframe into a csv file
clinic_df.to_csv("clinics_enriched.csv",index=False)


## Task 1.6 — Export the clean table and write the SQL load script

Sort the cleaned data by appointment_id, format both date columns as YYYY-MM-DD text (.dt.strftime("%Y-%m-%d")), and save appointments_clean.csv with exactly these 16 columns, in this order: appointment_id, patient_id, clinic_id, booked_on, appt_date, age, gender, distance_km, fee_inr, booking_channel, visit_type, reminder_sent, prior_no_shows, status, lead_days, no_show. Then, with a for loop and f-strings, write load_data.sql containing one statement per row, in this pattern (text in single quotes, numbers unquoted): INSERT INTO appointments VALUES ('A0001','P196','C05','2025-01-22',...,0); — and likewise one INSERT INTO clinics VALUES (...) per clinic from clinics_enriched.csv. Write all the clinics statements first, because the appointments table points to them with a foreign key. Before exporting, check and print that appointment_id is unique and that none of the 16 columns has a missing value. (Building SQL text with f-strings is fine for this fixed, controlled dataset — none of its values contains an apostrophe. It is not a safe method for real applications, where parameterised queries are used instead.)

In [ ]:
# Sort by appointment_id
df_clean_sorted = df_clean.sort_values(by='appointment_id')

# Format the date columns to YYYY-MM-DD
df_clean_sorted['booked_on'] = df_clean_sorted['booked_on'].dt.strftime("%Y-%m-%d")
df_clean_sorted['appt_date'] = df_clean_sorted['appt_date'].dt.strftime("%Y-%m-%d")

# Check if any columns have null values
print(f'Number of null values: \n {df_clean_sorted.isnull().sum()}')

# write the contnets into csv file
df_clean_sorted.to_csv('appointments_clean.csv',index=False)


# Read the clinics enriched csv
df_clinic = pd.read_csv('/content/clinics_enriched.csv')


def generate_insert_statements(df,table_name):
  """Helper function to dynamically generate INSERT statements for any DataFrame."""
  statements = []

  columns =  ",".join(df.columns)

  for _,row in df.iterrows():
    values = []
    for val in row:

      if isinstance(val, str):
        escape_val = val.replace("'","''")  # Even though in current data this is not there, adding it for making the program reusable
        values.append(f"'{escape_val}'")
      elif pd.isna(val):
        values.append("NULL")
      elif isinstance(val, bool):
        values.append(str(val).upper())
      else:
        values.append(str(val))

    values_str = ",".join(values)
    sql_str = f"INSERT INTO {table_name} ({columns}) VALUES ({values_str});"
    statements.append(sql_str)

  return statements

sql_filename = "load_data.sql"

with open(sql_filename, mode='w') as file:

  # Write the insert queries for the clinic dataframe
  clinic_sql_inserts = generate_insert_statements(df_clinic, 'clinics')
  file.write("-- Insert statements for Clinics\n")
  file.write("\n".join(clinic_sql_inserts) + "\n\n")

  # Write the insert queries for the clinic dataframe
  appts_sql_inserts = generate_insert_statements(df_clean_sorted, 'appointments')
  file.write("-- Insert statements for Appointments\n")
  file.write("\n".join(appts_sql_inserts) + "\n\n")

print(f"Successfully generated and wrote SQL statements to {sql_filename}")



Number of null values: 
 appointment_id     0
patient_id         0
clinic_id          0
booked_on          0
appt_date          0
age                0
gender             0
distance_km        0
fee_inr            0
booking_channel    0
visit_type         0
reminder_sent      0
prior_no_shows     0
status             0
lead_days          0
no_show            0
dtype: int64
Successfully generated and wrote SQL statements to load_data.sql
